In [6]:
# Installation libs
!pip install -q scikit-learn pandas numpy matplotlib seaborn lightgbm tldextract joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 3.3 MB/s eta 0:00:00


In [7]:
from google.colab import files
import io
import pandas as pd

# clique et choisis phishing_site_urls.csv depuis ton PC
uploaded = files.upload()


df = pd.read_csv(io.BytesIO(uploaded['phishing_site_urls.csv']), low_memory=False)
print("Loaded from upload:", df.shape)
df.head()


Saving phishing_site_urls.csv to phishing_site_urls.csv
Loaded from upload: (549346, 2)


,URL,Label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,bad
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,bad
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,bad
3,mail.printakid.com/www.online.americanexpress....,bad
4,thewhiskeydregs.com/wp-content/themes/widescre...,bad


In [8]:
# ---- Cellule : map labels en binaire et distribution ----
import pandas as pd
# df vient de ton upload ; on inspecte les colonnes affichées ("URL","Label" d'après ton output)
print(df.columns.tolist())
# On normalise les noms si nécessaire
df = df.rename(columns={df.columns[0]:'url', df.columns[1]:'label_raw'})
# map simple : 'bad' -> 1, tout le reste -> 0 (adaptable)
df['label'] = df['label_raw'].apply(lambda x: 1 if str(x).strip().lower()=='bad' else 0)
print("Total rows:", len(df))
print("Distribution labels (counts):")
print(df['label'].value_counts())
print("Distribution normalized:")
print(df['label'].value_counts(normalize=True))
df[['url','label_raw','label']].head(6)


['URL', 'Label']
Total rows: 549346
Distribution labels (counts):
label
0    392924
1    156422
Name: count, dtype: int64
Distribution normalized:
label
0    0.715258
1    0.284742
Name: proportion, dtype: float64


,url,label_raw,label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,bad,1
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,bad,1
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,bad,1
3,mail.printakid.com/www.online.americanexpress....,bad,1
4,thewhiskeydregs.com/wp-content/themes/widescre...,bad,1
5,smilesvoegol.servebbs.org/voegol.php,bad,1


In [9]:
# ---- Cellule : créer un échantillon stratifié ----
from sklearn.model_selection import train_test_split

# Taille d'échantillon à utiliser pour l'entraînement (ajuste selon RAM)
SAMPLE_SIZE = 120000  # ex : 120k ; si tu veux plus, augmente, mais attention mémoire
n_total = len(df)
if SAMPLE_SIZE >= n_total:
    df_sample = df.copy()
else:
    # stratified sample: on extrait SAMPLE_SIZE en respectant proportions
    frac = SAMPLE_SIZE / n_total
    # pandas >= 1.1 a sample(frac=...) ; pour stratify, on fait groupby.sample si pandas >= 1.1
    try:
        df_sample = df.groupby('label', group_keys=False).sample(frac=frac, random_state=42)
    except Exception as e:
        # fallback: use sklearn train_test_split by label groups
        df_sample, _ = train_test_split(df, train_size=SAMPLE_SIZE, stratify=df['label'], random_state=42)
print("Sample shape:", df_sample.shape)
print(df_sample['label'].value_counts(normalize=True))


Sample shape: (120000, 3)
label
0    0.715258
1    0.284742
Name: proportion, dtype: float64


In [10]:
# ---- Cellule : fonctions lexicales (réutilisables) ----
import re, math
from collections import Counter
import tldextract
import numpy as np

def has_ip(url):
    m = re.search(r'//(\d{1,3}\.){3}\d{1,3}', url)
    return 1 if m else 0

def str_entropy(s):
    if not s:
        return 0.0
    counts = Counter(s)
    probs = [c/len(s) for c in counts.values()]
    return -sum(p*math.log2(p) for p in probs)

def extract_lexical_features(url):
    try:
        ext = tldextract.extract(url)
        domain = ext.domain + ('.' + ext.suffix if ext.suffix else '')
    except:
        domain = ''
    features = {}
    features['url_len'] = len(url)
    features['domain_len'] = len(domain)
    features['num_dots'] = url.count('.')
    features['num_slash'] = url.count('/')
    features['num_params'] = url.count('?') + url.count('&')
    features['has_at'] = 1 if '@' in url else 0
    features['has_dash'] = 1 if '-' in domain else 0
    features['has_ip'] = has_ip(url)
    features['count_digits'] = sum(c.isdigit() for c in url)
    features['domain_entropy'] = str_entropy(domain)
    return features

# Appliquer sur l'échantillon (cela prend un peu de temps mais reste rapide)
lex_df = df_sample['url'].apply(extract_lexical_features).apply(pd.Series)
df_sample = pd.concat([df_sample.reset_index(drop=True), lex_df.reset_index(drop=True)], axis=1)
df_sample.head()


,url,label_raw,label,url_len,domain_len,num_dots,num_slash,num_params,has_at,has_dash,has_ip,count_digits,domain_entropy
0,depositaccounts.com/savings/,good,0,28.0,19.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,3.536887
1,citypages.com/related/to/Dave+Simonett/,good,0,39.0,13.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,3.546594
2,askart.com/askart/c/kate_carew/kate_carew.aspx,good,0,46.0,10.0,2.0,4.0,0.0,0.0,0.0,0.0,0.0,3.121928
3,brianwattsphoto.com/,good,0,20.0,19.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,3.642150
4,thefreedictionary.com/action+deferred,good,0,37.0,21.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,3.689704


In [11]:
# ---- Cellule : TF-IDF char n-grams ----
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3,5), max_features=3000)  # ajustable
X_tfidf = vectorizer.fit_transform(df_sample['url'])
print("TF-IDF shape:", X_tfidf.shape)


TF-IDF shape: (120000, 3000)


In [12]:
# ---- Cellule : assembler et split train/test ----
import scipy.sparse as sp
lex_features = ['url_len','domain_len','num_dots','num_slash','num_params','has_at','has_dash','has_ip','count_digits','domain_entropy']
X_lex = df_sample[lex_features].fillna(0).values
X = sp.hstack([X_tfidf, sp.csr_matrix(X_lex)]).tocsr()
y = df_sample['label'].astype(int).values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train / Test shapes:", X_train.shape, X_test.shape)


Train / Test shapes: (96000, 3010) (24000, 3010)


In [13]:
# ---- Cellule : entraîner LightGBM ----
!pip install -q lightgbm
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

lgb = LGBMClassifier(n_estimators=300, random_state=42)
lgb.fit(X_train, y_train)
y_pred = lgb.predict(X_test)
y_prob = lgb.predict_proba(X_test)[:,1]

print("Classification report:")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


[LightGBM] [Info] Number of positive: 27335, number of negative: 68665
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.653218 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 701020
[LightGBM] [Info] Number of data points in the train set: 96000, number of used features: 3009
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.284740 -> initscore=-0.921072
[LightGBM] [Info] Start training from score -0.921072


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Classification report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.97     17166
           1       0.95      0.92      0.93      6834

    accuracy                           0.96     24000
   macro avg       0.96      0.95      0.95     24000
weighted avg       0.96      0.96      0.96     24000

ROC-AUC: 0.991299482261234
Confusion matrix:
 [[16851   315]
 [  580  6254]]


In [14]:
# ---- Cellule : sauvegarder bundle (si LightGBM / TF-IDF utilisé) ----
import joblib
model_bundle = {
    'model': lgb,                # ou clf pour SGD
    'vectorizer': vectorizer,    # si TF-IDF
    'lex_features': lex_features
}
joblib.dump(model_bundle, 'phish_model_bundle.joblib')
print("Model bundle saved: phish_model_bundle.joblib")
# copier sur Drive si tu montes Drive :
# !cp phish_model_bundle.joblib /content/drive/MyDrive/


Model bundle saved: phish_model_bundle.joblib


In [15]:
# ---- Cellule : fonction utilitaire prédiction (pour LightGBM + TF-IDF) ----
import numpy as np
import scipy.sparse as sp
def predict_url(url, bundle):
    vec = bundle['vectorizer']
    model = bundle['model']
    lex_feats = bundle['lex_features']

    url_norm = url.strip().lower()
    x_tfidf = vec.transform([url_norm])
    lex = extract_lexical_features(url_norm)
    X_lex_single = np.array([[lex[f] for f in lex_feats]])
    X_single = sp.hstack([x_tfidf, sp.csr_matrix(X_lex_single)]).tocsr()

    prob_phish = float(model.predict_proba(X_single)[:,1][0])  # prob phishing
    label = "phishing" if prob_phish >= 0.5 else "legit"

    # calcul de la "confiance" selon le label prédit
    confidence = prob_phish if label == "phishing" else 1 - prob_phish

    return f"{label} ({confidence*100:.1f}% confident)"


In [16]:
#contenu selon num de ligne
i = 389542
print(f"URL: {df.iloc[i]['url']}  |  Label: {df.iloc[i]['label']}")

URL: miningweekly.com/article/argex-starts-pea-at-quebec-titanium-property-2011-07-04  |  Label: 0


In [21]:
# test rapide
bundle = joblib.load('phish_model_bundle.joblib')
print(predict_url("https://www.google.com/search?q=quantum+computer&sca_esv=7ea9874e2de762c7&sxsrf=AE3TifPp-4Ww51i7kF2cz_AmxPnA6n2avw%3A1760796682713&ei=CqDzaKeqK4fU7M8PiZDUgQg&udm=2&oq=quantu&gs_lp=Egxnd3Mtd2l6LXNlcnAaAhgCIgZxdWFudHUqAggBMhIQIxjwBRiABBgTGCcYyQIYigUyEBAjGPAFGIAEGCcYyQIYigUyChAjGPAFGCcYyQIyDRAAGIAEGLEDGEMYigUyCBAAGIAEGLEDMggQLhiABBixAzIFEAAYgAQyBRAAGIAEMgUQABiABDIOEAAYgAQYsQMYgwEYigVIlBdQAFiUBnAAeAGQAQCYAVqgAbUDqgEBNrgBAcgBAPgBAZgCBqACwQTCAgoQIxiABBgnGIoFwgIOEC4YgAQYxwEYjgUYrwHCAg4QLhiABBixAxjRAxjHAcICCxAAGIAEGLEDGIMBwgILEC4YgAQY0QMYxwHCAhEQLhiABBixAxjRAxiDARjHAcICChAAGIAEGEMYigXCAhMQLhiABBixAxjRAxhDGMcBGIoFwgIOEAAYgAQYkgMYuAQYigXCAgsQABiABBiSAxiKBcICCBAAGIAEGMkDwgIREC4YgAQYsQMYpAMYqAMYiwPCAhIQABiABBixAxhDGIoFGAoYiwPCAg0QLhiABBjRAxjHARgKmAMAkgcDNC4yoAedV7IHAzQuMrgHwQTCBwUzLTMuM8gHdQ&sclient=gws-wiz-serp", bundle))


legit (61.7% confident)


In [19]:
print(predict_url("https://www.youtube.com/watch?v=H8W9oMNSuwo&list=PLxbwE86jKRgMpuZuLBivzlM8s2Dk5lXBQ", bundle))

legit (97.8% confident)


In [20]:
print(predict_url("https://stackoverflow.com/questions/56604151/how-to-extract-multiple-objects-from-an-image-using-python-opencv", bundle))

legit (98.1% confident)


In [22]:
print(predict_url("www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrcmd=_home-customer&nav=1/loading.php", bundle))

phishing (100.0% confident)


In [23]:
print(predict_url("premierpaymentprocessing.com/includes/boleto-2via-07-2012.php", bundle))

phishing (99.2% confident)
